# 03 · LangGraph Deep Dive (Multi-Step ReAct)

## 🎯 What You'll Learn
- Multi-step reasoning chains with state persistence
- Real LangGraph StateGraph implementation
- Console logging and execution tracing
- Conditional edges and stop conditions
- Advanced workflow patterns for production agents

## 📋 Shared Legal Dataset
Using the same contracts for consistency:

```python
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."
CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."
```

Transform single-step ReAct into powerful multi-step reasoning workflows!


## Theory: Why Multi-Step ReAct with LangGraph?

**Single-Step Limitations:**
- One reasoning → one action → done
- No memory between tool calls
- Can't build on previous findings
- Limited for complex analysis

**Multi-Step Power:**
- **Chain reasoning**: Each step builds on the last
- **Persistent state**: Remember what we've learned
- **Conditional logic**: Different paths based on findings
- **Rich context**: Accumulate insights across steps

**LangGraph Architecture:**
```
START → reasoning_node → action_node → response_node → CONTINUE?
         ↑                                          ↓
         └──────────── (if not done) ←──────────────┘
```


In [ ]:
# Real LangGraph Implementation with Console Logging
from typing import Dict, List, TypedDict
from datetime import datetime
import json

# Shared dataset
CONTRACT_A = "EMPLOYMENT AGREEMENT between TechMahindra Solutions and John Doe. Termination: Either party may terminate with 30 days notice. Liability: Company liable for damages up to $50,000."
CONTRACT_B = "SERVICE AGREEMENT between ABC Corp and XYZ Ltd. Payment terms: Net 30 days. Governing law: This agreement shall be governed by Maharashtra state laws."

# Enhanced state management (mirrors our real app.py)
class AgentState(TypedDict):
    user_query: str
    step_count: int
    reasoning_chain: List[Dict[str, str]]
    context_memory: str
    current_action: str
    current_reasoning: str
    current_result: str
    is_complete: bool

def log_step(step_name: str, state: AgentState, details: str = ""):
    """Console logging like our real app"""
    timestamp = datetime.now().strftime("%H:%M:%S")
    step_num = state.get("step_count", 0)
    print(f"[{timestamp}] 📋 STEP {step_num}: {step_name}")
    print(f"[{timestamp}] 💭 REASONING: {state.get('current_reasoning', 'N/A')}")
    if details:
        print(f"[{timestamp}] 🔍 DETAILS: {details}")
    print(f"[{timestamp}] 🔄 CHAIN LENGTH: {len(state.get('reasoning_chain', []))} steps")
    print("-" * 60)

# Advanced legal tools (enhanced versions from our app)
def deep_search_tool(query: str, context: str = "") -> str:
    """Enhanced search with context awareness"""
    
    # Search both contracts
    contracts = {"CONTRACT_A": CONTRACT_A, "CONTRACT_B": CONTRACT_B}
    findings = []
    
    query_lower = query.lower()
    search_terms = ["liability", "termination", "payment", "governing", "damages", "risk"]
    
    for contract_name, contract_text in contracts.items():
        contract_lower = contract_text.lower()
        for term in search_terms:
            if term in query_lower and term in contract_lower:
                # Extract relevant sentence
                sentences = contract_text.split('. ')
                for sentence in sentences:
                    if term in sentence.lower():
                        findings.append(f"{contract_name}: {sentence.strip()}")
    
    if findings:
        result = f"[deep_search] Found {len(findings)} relevant clauses:\n" + "\n".join(findings[:3])
    else:
        result = f"[deep_search] Searched contracts for '{query}' - general contract analysis needed"
    
    return result

def risk_analyzer_tool(search_results: str, context: str = "") -> str:
    """Analyze risks based on search findings"""
    
    risk_indicators = {
        "liability": "HIGH RISK - Unlimited liability exposure",
        "damages": "MEDIUM RISK - Damage caps may apply", 
        "termination": "LOW RISK - Standard termination clause",
        "governing": "LOW RISK - Clear jurisdiction defined",
        "payment": "MEDIUM RISK - Payment terms need review"
    }
    
    risks_found = []
    for indicator, assessment in risk_indicators.items():
        if indicator in search_results.lower():
            risks_found.append(assessment)
    
    if risks_found:
        result = f"[risk_analyzer] Risk Assessment:\n" + "\n".join(risks_found)
    else:
        result = "[risk_analyzer] Overall risk: LOW - Standard contract terms identified"
    
    return result

def decision_maker_tool(risk_analysis: str, context: str = "") -> str:
    """Make final recommendation based on analysis"""
    
    if "HIGH RISK" in risk_analysis:
        recommendation = "❌ DO NOT SIGN - High risks identified. Negotiate liability caps."
    elif "MEDIUM RISK" in risk_analysis:
        recommendation = "⚠️ PROCEED WITH CAUTION - Review flagged terms before signing."
    else:
        recommendation = "✅ SAFE TO SIGN - Standard terms, minimal risk exposure."
    
    result = f"[decision_maker] Final Recommendation:\n{recommendation}\n\nBased on: {risk_analysis[:100]}..."
    return result

def compliance_checker_tool(analysis: str, context: str = "") -> str:
    """Check compliance requirements"""
    
    compliance_items = [
        "✓ Governing law clause present",
        "✓ Termination notice period specified", 
        "✓ Liability limitations defined",
        "? Payment terms clarity needs review"
    ]
    
    result = f"[compliance_checker] Compliance Check:\n" + "\n".join(compliance_items)
    return result

# Test the enhanced tools
print("🔧 TESTING ENHANCED LEGAL TOOLS")
print("=" * 50)

test_query = "What are the liability risks in this employment contract?"
search_result = deep_search_tool(test_query)
risk_result = risk_analyzer_tool(search_result)
decision_result = decision_maker_tool(risk_result)

print("Search Result:", search_result[:100] + "...")
print("Risk Analysis:", risk_result[:100] + "...")
print("Decision:", decision_result[:100] + "...")


## Multi-Step LangGraph Workflow

Now let's build the complete multi-step reasoning engine:


In [ ]:
# LangGraph-Style Workflow Implementation
class MultiStepLegalAgent:
    """Advanced multi-step reasoning agent with LangGraph patterns"""
    
    def __init__(self):
        self.max_steps = 4
        self.tools = {
            "deep_search": deep_search_tool,
            "risk_analyzer": risk_analyzer_tool,
            "compliance_checker": compliance_checker_tool,
            "decision_maker": decision_maker_tool
        }
    
    def reasoning_node(self, state: AgentState) -> AgentState:
        """Determine next action based on current state"""
        
        step_count = state.get("step_count", 0)
        user_query = state.get("user_query", "")
        context_memory = state.get("context_memory", "")
        
        # Multi-step reasoning logic (mirrors our app.py)
        if step_count == 0:
            # First step - always start with deep search
            reasoning = f"User asked: '{user_query}'. I need to start by gathering comprehensive information from the contract documents using deep_search."
            action = "deep_search"
            
        elif step_count == 1:
            # Second step - analyze risks based on findings
            reasoning = f"I found information in the previous step. Now I need to analyze the risk implications using risk_analyzer."
            action = "risk_analyzer"
            
        elif step_count == 2:
            # Third step - check compliance
            reasoning = f"I have risk analysis. Now I should check compliance requirements before making final decision."
            action = "compliance_checker"
            
        else:
            # Final step - make decision
            reasoning = f"I have gathered information, analyzed risks, and checked compliance. Time to make the final recommendation."
            action = "decision_maker"
        
        # Update state
        state["current_reasoning"] = reasoning
        state["current_action"] = action
        
        # Log the reasoning step
        log_step(f"REASONING (Step {step_count + 1})", state, f"Selected action: {action}")
        
        return state
    
    def action_node(self, state: AgentState) -> AgentState:
        """Execute the selected tool"""
        
        action = state.get("current_action", "")
        user_query = state.get("user_query", "")
        context_memory = state.get("context_memory", "")
        
        # Execute the tool
        if action in self.tools:
            try:
                # Use context for better results
                if action == "deep_search":
                    result = self.tools[action](user_query, context_memory)
                else:
                    # Pass previous results as context for subsequent tools
                    result = self.tools[action](context_memory, context_memory)
                
                state["current_result"] = result
                
                # Log the action execution
                log_step(f"ACTION - {action.upper()}", state, f"Tool executed successfully")
                
            except Exception as e:
                error_msg = f"Tool execution failed: {str(e)}"
                state["current_result"] = f"[error] {error_msg}"
                log_step(f"ACTION ERROR - {action.upper()}", state, error_msg)
        
        else:
            state["current_result"] = f"[error] Unknown action: {action}"
            log_step("ACTION ERROR", state, f"Unknown action: {action}")
        
        return state
    
    def response_node(self, state: AgentState) -> AgentState:
        """Update state and determine if we should continue"""
        
        # Add current step to reasoning chain
        reasoning_chain = state.get("reasoning_chain", [])
        
        current_step = {
            "step": len(reasoning_chain) + 1,
            "action": state.get("current_action", ""),
            "reasoning": state.get("current_reasoning", ""),
            "result": state.get("current_result", "")
        }
        
        reasoning_chain.append(current_step)
        state["reasoning_chain"] = reasoning_chain
        
        # Update context memory with latest result
        previous_context = state.get("context_memory", "")
        new_result = state.get("current_result", "")
        updated_context = f"{previous_context}\n\nStep {current_step['step']}: {new_result}".strip()
        state["context_memory"] = updated_context
        
        # Increment step count
        state["step_count"] = state.get("step_count", 0) + 1
        
        # Determine if we should continue
        step_count = state["step_count"]
        is_complete = (step_count >= self.max_steps) or ("decision_maker" in state.get("current_action", ""))
        state["is_complete"] = is_complete
        
        # Log the response update
        status = "COMPLETE" if is_complete else "CONTINUING"
        log_step(f"RESPONSE - {status}", state, f"Updated context memory ({len(updated_context)} chars)")
        
        return state
    
    def should_continue(self, state: AgentState) -> str:
        """Conditional edge logic"""
        is_complete = state.get("is_complete", False)
        step_count = state.get("step_count", 0)
        
        if is_complete or step_count >= self.max_steps:
            print(f"🏁 WORKFLOW COMPLETE - Stopping after {step_count} steps")
            return "end"
        else:
            print(f"🔄 WORKFLOW CONTINUING - Step {step_count + 1} next")
            return "continue"
    
    def run_workflow(self, user_query: str) -> AgentState:
        """Execute the complete multi-step workflow"""
        
        print(f"\n🚀 STARTING MULTI-STEP ANALYSIS")
        print(f"📝 Query: '{user_query}'")
        print("=" * 80)
        
        # Initialize state
        state: AgentState = {
            "user_query": user_query,
            "step_count": 0,
            "reasoning_chain": [],
            "context_memory": "",
            "current_action": "",
            "current_reasoning": "",
            "current_result": "",
            "is_complete": False
        }
        
        # Execute the workflow loop
        while not state.get("is_complete", False) and state.get("step_count", 0) < self.max_steps:
            # LangGraph node execution sequence
            state = self.reasoning_node(state)
            state = self.action_node(state) 
            state = self.response_node(state)
            
            # Check continuation condition
            if self.should_continue(state) == "end":
                break
        
        print(f"\n✅ WORKFLOW COMPLETED")
        print(f"📊 Total Steps: {len(state['reasoning_chain'])}")
        print(f"🧠 Final Context Length: {len(state.get('context_memory', ''))} characters")
        
        return state

# Create and test the agent
agent = MultiStepLegalAgent()

# Test with a complex legal query
test_result = agent.run_workflow("Explain me employment contract in simple words.")


## 🧪 Practice Exercise 1: Analyze the Reasoning Chain

Let's examine what the agent learned at each step:


In [ ]:
# Analyze the reasoning chain from our test
def analyze_reasoning_chain(result: AgentState):
    """Deep dive into the agent's reasoning process"""
    
    print("🔍 REASONING CHAIN ANALYSIS")
    print("=" * 60)
    
    reasoning_chain = result.get("reasoning_chain", [])
    
    for i, step in enumerate(reasoning_chain, 1):
        print(f"\n📋 STEP {i}: {step['action'].upper()}")
        print(f"🧠 Reasoning: {step['reasoning']}")
        print(f"📤 Result: {step['result'][:150]}...")
        
        # Analyze what the agent learned
        result_text = step['result'].lower()
        insights = []
        
        if 'liability' in result_text:
            insights.append("💡 Learned about liability terms")
        if 'risk' in result_text:
            insights.append("⚠️ Identified risk factors")
        if 'termination' in result_text:
            insights.append("📋 Found termination clauses")
        if 'recommendation' in result_text or 'sign' in result_text:
            insights.append("🎯 Made final recommendation")
        
        if insights:
            print("🔗 Key Insights: " + " | ".join(insights))
        
        print("-" * 50)
    
    # Overall analysis
    print(f"\n📊 OVERALL ANALYSIS:")
    print(f"• Total reasoning steps: {len(reasoning_chain)}")
    print(f"• Context accumulated: {len(result.get('context_memory', ''))} characters")
    print(f"• Tools used: {[step['action'] for step in reasoning_chain]}")
    
    # Check if it followed the expected pattern
    expected_pattern = ["deep_search", "risk_analyzer", "compliance_checker", "decision_maker"]
    actual_pattern = [step['action'] for step in reasoning_chain]
    
    if actual_pattern == expected_pattern:
        print("✅ Followed expected 4-step pattern perfectly!")
    else:
        print(f"⚠️ Pattern deviation: Expected {expected_pattern}, Got {actual_pattern}")

# Analyze our test result
analyze_reasoning_chain(test_result)


## 🧪 Practice Exercise 2: Custom Workflow Patterns

Build your own multi-step workflow with different stop conditions:


In [ ]:
class CustomWorkflowAgent(MultiStepLegalAgent):
    """
    YOUR CHALLENGE: Customize the workflow with different patterns!
    
    Ideas to implement:
    1. Early stopping if high risk detected
    2. Dynamic tool selection based on query type
    3. Conditional branching (different paths for different contract types)
    4. Confidence scoring and retry logic
    """
    
    def __init__(self, max_steps: int = 5):
        super().__init__()
        self.max_steps = max_steps
        self.confidence_threshold = 0.8
    
    def reasoning_node(self, state: AgentState) -> AgentState:
        """Enhanced reasoning with dynamic tool selection"""
        
        step_count = state.get("step_count", 0)
        user_query = state.get("user_query", "").lower()
        context_memory = state.get("context_memory", "")
        
        # Dynamic reasoning based on query type and context
        if step_count == 0:
            # Smart first step selection
            if any(term in user_query for term in ["payment", "money", "fee"]):
                reasoning = "Query is about payment terms. Starting with targeted search."
                action = "deep_search"
            elif any(term in user_query for term in ["risk", "dangerous", "safe"]):
                reasoning = "Risk-focused query. Starting with comprehensive search."
                action = "deep_search"
            else:
                reasoning = "General contract query. Starting with broad search."
                action = "deep_search"
                
        elif step_count == 1:
            # Adaptive second step
            if "HIGH RISK" in context_memory:
                reasoning = "High risk detected! Prioritizing immediate risk analysis."
                action = "risk_analyzer"
            else:
                reasoning = "Standard risk analysis based on search findings."
                action = "risk_analyzer"
                
        elif step_count == 2:
            # Conditional third step
            if "HIGH RISK" in context_memory:
                reasoning = "High risk confirmed. Skipping compliance, going straight to decision."
                action = "decision_maker"  # Skip compliance for high-risk cases
            else:
                reasoning = "Moderate/low risk. Checking compliance before decision."
                action = "compliance_checker"
                
        else:
            # Final decision
            reasoning = "All analysis complete. Making final recommendation."
            action = "decision_maker"
        
        # Update state
        state["current_reasoning"] = reasoning
        state["current_action"] = action
        
        # Enhanced logging
        risk_level = "HIGH" if "HIGH RISK" in context_memory else "MODERATE/LOW"
        log_step(f"CUSTOM REASONING (Step {step_count + 1})", state, 
                f"Risk Level: {risk_level}, Selected: {action}")
        
        return state
    
    def should_continue(self, state: AgentState) -> str:
        """Enhanced stop conditions"""
        
        step_count = state.get("step_count", 0)
        context_memory = state.get("context_memory", "")
        current_action = state.get("current_action", "")
        
        # Early stopping conditions
        if "HIGH RISK" in context_memory and current_action == "decision_maker":
            print(f"🚨 EARLY STOP - High risk detected, decision made at step {step_count}")
            return "end"
        
        # Standard stopping conditions
        if step_count >= self.max_steps:
            print(f"🏁 MAX STEPS REACHED - Stopping at {step_count} steps")
            return "end"
        
        if current_action == "decision_maker":
            print(f"🎯 DECISION MADE - Workflow complete at step {step_count}")
            return "end"
        
        print(f"🔄 CONTINUING - Next step {step_count + 1}")
        return "continue"

# Test different workflow patterns
print("🧪 TESTING CUSTOM WORKFLOW PATTERNS")
print("=" * 70)

custom_agent = CustomWorkflowAgent(max_steps=5)

# Test 1: High-risk scenario (should trigger early stopping)
print("\n🔥 TEST 1: High-Risk Scenario")
result1 = custom_agent.run_workflow("What are the liability risks in this contract?")

print("\n" + "="*50)

# Test 2: Payment-focused query
print("\n💰 TEST 2: Payment-Focused Query") 
result2 = custom_agent.run_workflow("When do I need to pay under this service agreement?")

# Compare the workflows
def compare_workflows(result1, result2, labels):
    """Compare two workflow executions"""
    print(f"\n📊 WORKFLOW COMPARISON")
    print("=" * 50)
    
    for i, (result, label) in enumerate(zip([result1, result2], labels)):
        chain = result.get("reasoning_chain", [])
        actions = [step["action"] for step in chain]
        print(f"\n{label}:")
        print(f"  Steps: {len(chain)}")
        print(f"  Path: {' → '.join(actions)}")
        print(f"  Context: {len(result.get('context_memory', ''))} chars")

compare_workflows(result1, result2, ["High-Risk Query", "Payment Query"])


## 📊 Final Assessment & Key Takeaways

**Acceptance Criteria for Notebook 3:**
- ✅ You understand multi-step vs single-step ReAct patterns
- ✅ You can implement LangGraph-style state management
- ✅ You've built custom workflow logic with conditional edges
- ✅ You can trace and debug multi-step reasoning chains
- ✅ You understand when to use different stop conditions

**What You've Mastered:**
1. **Stateful Workflows**: Persistent state across multiple reasoning steps
2. **Node Architecture**: Separation of reasoning, action, and response logic
3. **Conditional Edges**: Dynamic workflow paths based on intermediate results
4. **Context Accumulation**: Building rich context memory across steps
5. **Advanced Patterns**: Early stopping, dynamic tool selection, custom workflows

**Production Patterns Learned:**
- **Console Logging**: Real-time visibility into agent reasoning
- **State Persistence**: Maintaining context across complex workflows  
- **Error Handling**: Graceful degradation in multi-step scenarios
- **Performance Monitoring**: Tracking step counts and context size
- **Workflow Customization**: Adapting behavior based on query types

**Connection to Full Project:**
- This IS the core architecture of our main `app.py` agent!
- Same `AgentState` structure with `reasoning_chain` and `context_memory`
- Same node pattern: `reasoning_node` → `action_node` → `response_node`
- Same conditional logic for multi-step continuation
- Same console logging pattern for debugging

**Next Up:** Notebook 4 - RAG Integration with real document processing!
